In [ ]:
# !pip install -qU cohere

In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_community.vectorstores import FAISS

from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-11")

In [ ]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}:\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [ ]:
documents = TextLoader("./data/appendix-keywords.txt").load()

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

texts = text_splitter.split_documents(documents)

In [ ]:
embeddings = CohereEmbeddings(model="embed-multilingual-v3.0")

In [ ]:
retriever = FAISS.from_documents(
    texts, embeddings
).as_retriever(search_kwargs={"k": 10})

In [ ]:
query = "Word2Vec 에 대해서 알려줘!"

In [ ]:
docs = retriever.invoke(query)

In [ ]:
pretty_print_docs(docs)

CohereRerank 시용

In [ ]:
compressor = CohereRerank(model="rerank-multilingual-v3.0")  # 문서 재정렬 모델

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=retriver
)

In [ ]:
compressed_docs = compression_retreiver.invoke("Word2Vec 에 대해서 알려줘!")

In [ ]:
pretty_print_docs(compressed_docs)